# Research Assistant: run in Google Colab
Download `research_assistant_portfolio.zip` from the project chat, open this notebook in Colab, and run cells in order. This notebook imports the same project code; it does not copy or replace the application.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import zipfile

print("Upload the GitHub repository ZIP or research_assistant_portfolio.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]
assert len(zip_names) == 1, "Upload exactly one project ZIP file."
staging = Path("/content/project_upload")
staging.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as z:
    z.extractall(staging)
roots = list(staging.glob("*/pyproject.toml"))
assert len(roots) == 1, "Expected one project directory containing pyproject.toml."
source = roots[0].parent
project = Path("/content/research_assistant")
if source != project:
    shutil.copytree(source, project, dirs_exist_ok=True)
print("Project ready:", project)


In [ ]:
%cd /content/research_assistant
%pip install -e ".[dev]"

# Check that the package is visible to the notebook kernel.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("/content/research_assistant/src")))
import research_assistant.core
print("Import OK:", research_assistant.core.__file__)


In [ ]:
!python -m pytest -q
!python eval/run_eval.py


## Ask the included example PDF
PDF page numbers are one-based PDF pages. A citation is a prompt to read the passage, not proof that every claim is supported.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, "/content/research_assistant/src")
from research_assistant.core import Retriever, answer, ingest_pdf

sample = Path("/content/research_assistant/sample_data/sample_study.pdf")
passages = ingest_pdf(sample.name, sample.read_bytes())
retriever = Retriever(passages)

def ask(question, mode="extractive", api_key=None, model=None):
    result = answer(question, retriever, mode=mode, api_key=api_key, model=model)
    print("Answer:", result.text)
    for hit in result.hits:
        print(f"\n[{hit.id}] {hit.passage.document}, PDF page {hit.passage.page}:")
        print(hit.passage.text)
    return result

ask("How much did treatment reduce pain?")
ask("What was the satellite orbital altitude?")


## Use your own PDF(s)
Run this cell and upload selectable-text research PDFs (up to 20 MB each). Recreate the retriever after every new upload.

In [ ]:
print("Upload one or more PDFs")
pdfs = files.upload()
passages = []
for name, data in pdfs.items():
    try:
        passages.extend(ingest_pdf(name, data))
        print("Indexed:", name)
    except ValueError as exc:
        print("Skipped:", name, "—", exc)
if passages:
    retriever = Retriever(passages)
    print("Passages indexed:", len(passages))
    ask(input("Question about your PDFs: "))


## Optional: use an LLM
The default endpoint is the OpenAI-compatible `/chat/completions` API. This sends the question and retrieved PDF passages to the provider and may incur charges. Run after loading a PDF above. Avoid sensitive papers.

In [ ]:
from getpass import getpass
# Uncomment these lines when you have a provider key and model ID:
# api_key = getpass("API key (hidden): ")
# model = input("Model ID: ").strip()
# ask(input("Question: "), mode="llm", api_key=api_key, model=model)
